# 1. Installing and Importing the Required Libraries

In [2]:
%pip install numpy pandas scipy requests beautifulsoup4 lxml matplotlib seaborn scikit-learn xgboost joblib plotly streamlit pydeck shapely pyproj astropy astroquery skyfield tqdm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   -- ------------------------------------- 2.4/37.4 MB 13.8 MB/s eta 0:00:03
   ----- ---------------------------------- 5.5/37.4 MB 15.2 MB/s eta 0:00:03
   --------- ------------------------------ 9.2/37.4 MB 15.7 MB/s eta 0:00:02
   ------------- -------------------------- 12.6/37.4 MB 15.9 MB/s eta 0:00:02
   ----------------- ---------------------- 16.3/37.4 MB 16.0 MB/s eta 0:00:02
   --------------------- ------------------ 19.7/37.4 MB 16.0 MB/s eta 0:00:02
   ------------------------ --------------- 22.8/37.4 MB 16.1 MB/s eta 0:00:01
   ---------------------------- ----------- 26.2/37.4 MB 16.1 MB/s eta 0:00:01
   ------------------------------- -------- 29.9/37.4 MB 16.1 MB/s eta 0:00:01
   ----------------------------------- ---- 33.0/37.4 MB 16.1 MB/s eta 0:00:01
   ---------------------------------------  36.4/37.4 MB 16.1 MB/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [3]:
import numpy as np
import pandas as pd
import scipy
import requests
import bs4
import lxml
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import xgboost
import joblib
import plotly.express as px
import streamlit as st
import pydeck as pdk
import shapely
import pyproj
import astropy
import astroquery
import skyfield
import tqdm

print("All libraries imported successfully!")

All libraries imported successfully!


# 2. Data Extraction

## Dataset A
We check the NASA Five Millennium Catalog found in <https://eclipse.gsfc.nasa.gov/SEpubs/5MKSE.html> and find the ASCII text format which suits better for the extraction since it provides the complete set of 11,898 eclipse records in a compact, machine-readable file. According to NASA, the ASCII file occupies approximately 1.3 MB, compared with 13.2 MB for the PDF publication. This reduces the amount of data transferred by approximately 90% and avoids unnecessary document parsing. \
**Note: The data has errors that must be taken into consideration as shown below:**

### Dataset Integrity and Errata Verification

During the data acquisition process, an official NASA
errata notice was identified for the Five Millennium
Catalog of Solar Eclipses.

The notice reports incorrect tabulated values of
Delta T and geographic longitude in pages A-56 to
A-150 of the original printed publication.

NASA confirms that these corrections are included
in the PDF version of the catalog. However, the
notice does not explicitly confirm whether the
machine-readable ASCII and CSV datasets contain
the corrected values.

To address this uncertainty, the following
validation procedure will be implemented:

1. Compare overlapping numerical variables between
   the ASCII catalog and the Besselian Elements CSV.
2. Identify discrepancies exceeding the expected
   numerical rounding precision.
3. Verify potentially affected records against the
   corrected NASA publication.
4. Apply only documented and verified corrections.
5. Preserve the original datasets and maintain
   a record of all modifications.

This procedure is intended to prevent known data
quality issues from propagating into the machine
learning models, while ensuring reproducibility
and traceability.

**Reference:**
[NASA Five Millennium Catalog Errata](https://eclipse.gsfc.nasa.gov/5MCSE/5MKSEerrata.html)

In [12]:
url_a = "https://eclipse.gsfc.nasa.gov/5MCSE/5MKSEcatalog.txt"

response = requests.get(url_a, timeout=60)
response.raise_for_status()

# 1. Define the columns
columns_a = [
    "cat_no", "canon_plate", "year", "month", "day",
    "td_ge", "dt", "luna_num", "saros", "eclipse_type",
    "qle", "gamma", "magnitude", "lat_ge", "lng_ge",
    "sun_alt", "sun_azm", "path_width", "central_duration"
]

# 2. Extract the eclipse records
rows = []

text = response.content.decode("latin-1")

for line in text.splitlines():
    values = line.split()

    if len(values) >= 17 and values[0].isdigit(): #There are some columns with missing values we do this procedure to make sure that all rows are included.
        if values[9][0] in ["P", "A", "T", "H"]:

            while len(values) < len(columns_a):
                values.append(None)

            rows.append(values[:len(columns_a)])

# 3. Create the DataFrame
df_a = pd.DataFrame(rows, columns=columns_a)

# 4. Convert numeric columns
numeric_a = [
    "cat_no", "canon_plate", "year", "day", "dt",
    "luna_num", "saros", "gamma", "magnitude",
    "sun_alt", "sun_azm", "path_width"
]

for col in numeric_a:
    df_a[col] = pd.to_numeric(df_a[col], errors="coerce") #We make sure that all the numerical columns correspond to a numeric type value.

# 5. Validate and save
print("Dataset shape:", df_a.shape)

if len(df_a) != 11898:
    raise ValueError("The catalog is incomplete.")

df_a.set_index("cat_no", inplace=True)
display(df_a.head())

df_a.to_csv("dataset_A.csv", index=False)

Dataset shape: (11898, 19)


C:\Users\andre\AppData\Local\Temp\ipykernel_12996\2615221653.py:49: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_a.set_index("cat_no", inplace=True)


,canon_plate,year,month,day,td_ge,dt,luna_num,saros,eclipse_type,qle,gamma,magnitude,lat_ge,lng_ge,sun_alt,sun_azm,path_width,central_duration
cat_no,,,,,,,,,,,,,,,,,,
1,1,-1999,Jun,12,03:14:51,46438,-49456,5,T,-n,-0.2701,1.0733,6.0N,33.3W,74,344,247.0,06m37s
2,1,-1999,Dec,5,23:45:23,46426,-49450,10,A,n-,-0.2317,0.9382,32.9S,10.8E,76,21,236.0,06m44s
3,1,-1998,Jun,1,18:09:16,46415,-49444,15,T,p-,0.4994,1.0284,46.2N,83.4E,60,151,111.0,02m15s
4,1,-1998,Nov,25,05:57:03,46403,-49438,20,A,p-,-0.9045,0.9806,67.8S,143.8W,25,74,162.0,01m14s
5,1,-1997,Apr,22,13:19:56,46393,-49433,-13,P,-t,-1.4670,0.1611,60.6S,106.4W,0,281,NaN,NaN


## Dataset B
We go again to <https://eclipse.gsfc.nasa.gov/SEpubs/5MKSE.html> and get the Catalog of Solar Eclipse Besselian elements in CSV format. The CSV format was selected because it provides the numerical coefficients in a structured representation that can be directly imported into pandas. **Note: Since this dataset shares several variables with Dataset A, overlapping information will be evaluated to avoid unnecessary data redundancy.**

In [13]:
#We follow the same procedure to extract dataset B
# 1. Data source
url_b = "https://eclipse.gsfc.nasa.gov/eclipse_besselian_from_mysqldump2.csv"

# 2. Extract the data
df_b = pd.read_csv(url_b, low_memory=False)
# 3. Clean column names
df_b.columns = df_b.columns.str.strip()
# 4. Validate the dataset
print("Dataset shape:", df_b.shape)
if len(df_b) != 11898:
    raise ValueError("The Besselian catalog is incomplete.")
display(df_b.head())
# 5. Save the dataset
df_b.to_csv("dataset_B.csv", index=False)


Dataset shape: (11898, 54)


,year,month,day,td_ge,dt,luna_num,saros,eclipse_type,gamma,magnitude,...,tan_f2,tmin,tmax,etype,PNS,UNS,NCN,nSer,nSeq,nJLE
0,-1999,6,12,03:14:51,46438.2,-49456,5,T,-0.27009,1.07329,...,0.004578,-3.0,3.0,1,0,0,0,73,41,4
1,-1999,12,5,23:45:23,46426.5,-49450,10,A,-0.23172,0.93818,...,0.004732,-3.0,3.0,2,0,0,0,73,27,40
2,-1998,6,1,18:09:16,46414.6,-49444,15,T,0.49936,1.02844,...,0.004573,-3.0,3.0,1,1,0,0,75,32,20
3,-1998,11,25,05:57:03,46402.8,-49438,20,A,-0.90454,0.98056,...,0.004737,-3.0,3.0,2,-1,0,0,72,17,20
4,-1997,4,22,13:19:56,46392.9,-49433,-13,P,-1.46705,0.16108,...,0.004576,-3.0,3.0,4,-1,-1,-1,73,72,1


## Dataset C
In this case the information comes from <https://eclipse.gsfc.nasa.gov/SEpath/SEpath.html>. This page offers tables with a column called "Path Table" which gives you the geo coordinates of the shadow and its features (path width, central line duration etc.). 

In [15]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 1. Main NASA page
url_c = "https://eclipse.gsfc.nasa.gov/SEpath/SEpath.html"


# 2. Convert coordinates to decimal degrees
def read_coordinate(parts, i):

    if parts[i] == "-":
        return float("nan"), i + 1

    degrees = float(parts[i])
    minutes = float(parts[i + 1][:-1])
    direction = parts[i + 1][-1]

    if direction not in ["N", "S", "E", "W"]:
        raise ValueError("Invalid coordinate")

    coordinate = degrees + minutes / 60

    if direction in ["S", "W"]:
        coordinate = -coordinate

    return coordinate, i + 2


# 3. Extract the path of one eclipse
def extract_path(url):

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    event_id = url.split("/")[-1].replace("path.html", "")

    rows = []

    for pre in soup.find_all("pre"):

        for line in pre.get_text().splitlines():

            parts = line.split()
            if not parts:
                continue

            time = parts[0]

            if len(time) != 5 or time[2] != ":":
                continue

            if not (time[:2].isdigit() and time[3:].isdigit()):
                continue

            try:
                coordinates = []
                i = 1

                for j in range(6):
                    value, i = read_coordinate(parts, i)
                    coordinates.append(value)

                extra = parts[i:]

                if len(extra) < 5:
                    continue

                row = {
                    "event_id": event_id,
                    "time_ut": parts[0],
                    "north_lat": coordinates[0],
                    "north_lon": coordinates[1],
                    "south_lat": coordinates[2],
                    "south_lon": coordinates[3],
                    "center_lat": coordinates[4],
                    "center_lon": coordinates[5],
                    "diameter_ratio": pd.to_numeric(extra[0], errors="coerce"),
                    "sun_alt": pd.to_numeric(extra[1], errors="coerce"),
                    "sun_azm": pd.to_numeric(extra[2], errors="coerce"),
                    "path_width_km": pd.to_numeric(extra[3], errors="coerce"),
                    "central_duration": extra[4]
                }

                rows.append(row)

            except (ValueError, IndexError):
                continue

    return pd.DataFrame(rows)


# 4. Find the eclipse path links
response = requests.get(url_c, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

path_urls = []

for link in soup.find_all("a", href=True):  # Since the webpage has multiple tables that we need to access through different links.

    if link.get_text(strip=True).lower() == "path":
        path_url = urljoin(url_c, link["href"])
        path_urls.append(path_url)

path_urls = list(dict.fromkeys(path_urls))

print("Paths found:", len(path_urls))


# 5. Extract all eclipse paths
path_data = []

for i, url in enumerate(path_urls, 1):

    try:
        df_path = extract_path(url)

        if not df_path.empty:
            path_data.append(df_path)
        else:
            print("No data:", url)

    except requests.RequestException as error:
        print("Error:", url, error)

    print(f"Processed: {i}/{len(path_urls)}")
    time.sleep(0.2)


# 6. Combine and save the dataset
if not path_data:
    raise ValueError("No eclipse paths were extracted.")

df_c = pd.concat(path_data, ignore_index=True)

df_c.to_csv("dataset_C.csv", index=False)

print("Dataset C:", df_c.shape)
print("Eclipses extracted:", df_c["event_id"].nunique())

display(df_c.head(20))


Paths found: 88
Processed: 1/88
Processed: 2/88
Processed: 3/88
Processed: 4/88
Processed: 5/88
Processed: 6/88
Processed: 7/88
Processed: 8/88
Processed: 9/88
Processed: 10/88
Processed: 11/88
Processed: 12/88
Processed: 13/88
Processed: 14/88
Processed: 15/88
Processed: 16/88
Processed: 17/88
Processed: 18/88
Processed: 19/88
Processed: 20/88
Processed: 21/88
Processed: 22/88
Processed: 23/88
Processed: 24/88
Processed: 25/88
Processed: 26/88
Processed: 27/88
Processed: 28/88
Processed: 29/88
Processed: 30/88
Processed: 31/88
Processed: 32/88
Processed: 33/88
Processed: 34/88
Processed: 35/88
Processed: 36/88
Processed: 37/88
Processed: 38/88
Processed: 39/88
Processed: 40/88
Processed: 41/88
Processed: 42/88
Processed: 43/88
Processed: 44/88
Processed: 45/88
Processed: 46/88
Processed: 47/88
Processed: 48/88
Processed: 49/88
Processed: 50/88
Processed: 51/88
Processed: 52/88
Processed: 53/88
Processed: 54/88
Processed: 55/88
Processed: 56/88
Processed: 57/88
Processed: 58/88
Process

,event_id,time_ut,north_lat,north_lon,south_lat,south_lon,center_lat,center_lon,diameter_ratio,sun_alt,sun_azm,path_width_km,central_duration
0,SE1991Jan15A,22:00,-31.875000,119.320000,NaN,NaN,-30.723333,111.783333,0.915,2.0,113.0,358.0,05m41.0s
1,SE1991Jan15A,22:01,-32.873333,122.036667,NaN,NaN,-32.915000,117.513333,0.916,8.0,110.0,350.0,05m50.2s
2,SE1991Jan15A,22:02,-33.648333,124.213333,-33.796667,115.200000,-34.026667,120.553333,0.917,11.0,108.0,346.0,05m55.5s
3,SE1991Jan15A,22:03,-34.295000,126.086667,-35.106667,118.786667,-34.858333,122.908333,0.918,13.0,107.0,343.0,05m59.8s
4,SE1991Jan15A,22:04,-34.853333,127.755000,-36.020000,121.363333,-35.538333,124.901667,0.918,15.0,106.0,340.0,06m03.5s


In [16]:
display(df_c.head(20))

,event_id,time_ut,north_lat,north_lon,south_lat,south_lon,center_lat,center_lon,diameter_ratio,sun_alt,sun_azm,path_width_km,central_duration
0,SE1991Jan15A,22:00,-31.875000,119.320000,NaN,NaN,-30.723333,111.783333,0.915,2.0,113.0,358.0,05m41.0s
1,SE1991Jan15A,22:01,-32.873333,122.036667,NaN,NaN,-32.915000,117.513333,0.916,8.0,110.0,350.0,05m50.2s
2,SE1991Jan15A,22:02,-33.648333,124.213333,-33.796667,115.200000,-34.026667,120.553333,0.917,11.0,108.0,346.0,05m55.5s
3,SE1991Jan15A,22:03,-34.295000,126.086667,-35.106667,118.786667,-34.858333,122.908333,0.918,13.0,107.0,343.0,05m59.8s
4,SE1991Jan15A,22:04,-34.853333,127.755000,-36.020000,121.363333,-35.538333,124.901667,0.918,15.0,106.0,340.0,06m03.5s
5,SE1991Jan15A,22:05,-35.346667,129.275000,-36.750000,123.540000,-36.121667,126.661667,0.919,17.0,104.0,337.0,06m06.8s
6,SE1991Jan15A,22:06,-35.790000,130.680000,-37.365000,125.415000,-36.633333,128.256667,0.919,18.0,103.0,335.0,06m10.0s
7,SE1991Jan15A,22:07,-36.191667,131.993333,-37.900000,127.101667,-37.090000,129.726667,0.919,20.0,102.0,332.0,06m12.9s
8,SE1991Jan15A,22:08,-36.558333,133.230000,-38.375000,128.650000,-37.503333,131.096667,0.920,21.0,101.0,330.0,06m15.7s
9,SE1991Jan15A,22:09,-36.896667,134.405000,-38.801667,130.086667,-37.880000,132.386667,0.920,22.0,100.0,328.0,06m18.3s


## Dataset D
For this dataset we use the Horizon API found in the link <https://ssd.jpl.nasa.gov/horizons/> which allows the extraction of ephemeris data for specific celestial bodies, time intervals, reference frames, and temporal resolutions.

In [3]:

import pandas as pd
import requests
import time
from io import StringIO

# =========================================================
# DATASET D — JPL HORIZONS API
# =========================================================

# 1. Load Dataset C
df_c = pd.read_csv("dataset_C.csv")

url_d = "https://ssd.jpl.nasa.gov/api/horizons.api"


# 2. Add complete dates to Dataset C
events = []

for event_id, group in df_c.groupby("event_id", sort=False):

    group = group.copy()

    # Get the eclipse date from event_id
    event_date = pd.to_datetime(
        event_id[2:11], format="%Y%b%d"
    )

    # Convert HH:MM into time values
    clock = pd.to_timedelta(group["time_ut"] + ":00")

    # Detect when the trajectory crosses midnight
    new_day = (clock.diff() < pd.Timedelta(0)).cumsum()

    full_time = (
        event_date
        + clock
        + pd.to_timedelta(new_day, unit="D")
    )

    # Align the middle of the trajectory with the event date
    middle_date = full_time.iloc[len(group) // 2].normalize()

    full_time = full_time + (event_date - middle_date)

    group["datetime_ut"] = full_time

    events.append(group)

df_c = pd.concat(events, ignore_index=True)


# 3. Extract vectors from JPL Horizons
def get_horizons(body_id, start, stop):

    params = {
        "format": "text",
        "COMMAND": f"'{body_id}'",
        "OBJ_DATA": "'NO'",
        "MAKE_EPHEM": "'YES'",
        "EPHEM_TYPE": "'VECTORS'",
        "CENTER": "'500@399'",
        "REF_PLANE": "'FRAME'",
        "REF_SYSTEM": "'ICRF'",
        "OUT_UNITS": "'KM-S'",
        "VEC_TABLE": "'2'",
        "VEC_LABELS": "'NO'",
        "VEC_CORR": "'NONE'",
        "CSV_FORMAT": "'YES'",
        "TIME_TYPE": "'UT'",
        "TIME_DIGITS": "'MINUTES'",
        "STEP_SIZE": "'1 m'",
        "START_TIME": f"'{start}'",
        "STOP_TIME": f"'{stop}'"
    }

    response = requests.get(
        url_d, params=params, timeout=90
    )

    response.raise_for_status()

    text = response.text

    # 4. Extract the data section
    if "$$SOE" not in text or "$$EOE" not in text:
        raise ValueError("No ephemeris data found: " + text[:300])

    data = text.split("$$SOE")[1]
    data = data.split("$$EOE")[0]

    # 5. Create the DataFrame
    columns = [
        "jd_ut", "date_ut",
        "x_km", "y_km", "z_km",
        "vx_kms", "vy_kms", "vz_kms"
    ]

    df = pd.read_csv(
        StringIO(data),
        header=None,
        names=columns,
        usecols=range(8),
        skipinitialspace=True
    )

    # 6. Convert the dates
    df["date_ut"] = df["date_ut"].str.replace(
        "A.D.", "", regex=False
    )

    df["date_ut"] = df["date_ut"].str.strip()

    df["datetime_ut"] = pd.to_datetime(
        df["date_ut"],
        format="%Y-%b-%d %H:%M:%S.%f"
    )

    # 7. Convert numerical columns
    for col in columns:
        if col != "date_ut":
            df[col] = pd.to_numeric(df[col], errors="raise")

    # 8. Verify timestamps have minute precision
    if not (df["datetime_ut"] == df["datetime_ut"].dt.floor("min")).all():
        raise ValueError("Unexpected timestamp precision.")

    # 9. Remove unnecessary columns
    df = df.drop(columns=["jd_ut", "date_ut"])

    return df


# 10. Extract Sun and Moon vectors for every eclipse
path_data = []

for event_id, group in df_c.groupby("event_id", sort=False):

    # Observation interval
    start = group["datetime_ut"].min()
    stop = group["datetime_ut"].max() + pd.Timedelta(minutes=1)

    start = start.strftime("%Y-%m-%d %H:%M")
    stop = stop.strftime("%Y-%m-%d %H:%M")

    # Extract Sun and Moon data
    df_sun = get_horizons("10", start, stop)
    df_moon = get_horizons("301", start, stop)

    # Combine both bodies at the same time
    df_vectors = pd.merge(
        df_sun,
        df_moon,
        on="datetime_ut",
        suffixes=("_sun", "_moon"),
        validate="one_to_one"
    )

    # Keep the exact observations from Dataset C
    df_event = pd.merge(
        group[["event_id", "time_ut", "datetime_ut"]],
        df_vectors,
        on="datetime_ut",
        how="left",
        validate="one_to_one"
    )

    # Verify the extraction
    if df_event.isnull().any().any():
        raise ValueError(f"Missing observations: {event_id}")

    path_data.append(df_event)

    print("Completed:", event_id)

    time.sleep(0.2)


# 11. Combine all eclipses
if not path_data:
    raise ValueError("No eclipse data was extracted.")

df_d = pd.concat(path_data, ignore_index=True)


# 12. Final validation
if len(df_d) != len(df_c):
    raise ValueError("Dataset D does not match Dataset C.")

if df_d["event_id"].nunique() != df_c["event_id"].nunique():
    raise ValueError("Some eclipses are missing.")

if df_d.duplicated(["event_id", "datetime_ut"]).any():
    raise ValueError("Duplicated observations found.")


# 13. Save Dataset D
df_d.to_csv("dataset_D.csv", index=False)

print("Dataset D:", df_d.shape)
print("Unique eclipses:", df_d["event_id"].nunique())

display(df_d.head())


Completed: SE1991Jan15A
Completed: SE1991Jul11T
Completed: SE1992Jan04A
Completed: SE1992Jun30T
Completed: SE1994May10A
Completed: SE1994Nov03T
Completed: SE1995Apr29A
Completed: SE1995Oct24T
Completed: SE1997Mar09T
Completed: SE1998Feb26T
Completed: SE1998Aug22A
Completed: SE1999Feb16A
Completed: SE1999Aug11T
Completed: SE2001Jun21T
Completed: SE2001Dec14A
Completed: SE2002Jun10A
Completed: SE2002Dec04T
Completed: SE2003May31A
Completed: SE2003Nov23T
Completed: SE2005Apr08H
Completed: SE2005Oct03A
Completed: SE2006Mar29T
Completed: SE2006Sep22A
Completed: SE2008Feb07A
Completed: SE2008Aug01T
Completed: SE2009Jan26A
Completed: SE2009Jul22T
Completed: SE2010Jan15A
Completed: SE2010Jul11T
Completed: SE2012May20A
Completed: SE2012Nov13T
Completed: SE2013May10A
Completed: SE2013Nov03H
Completed: SE2014Apr29A
Completed: SE2015Mar20T
Completed: SE2016Mar09T
Completed: SE2016Sep01A
Completed: SE2017Feb26A
Completed: SE2017Aug21T
Completed: SE2019Jul02T
Completed: SE2019Dec26A
Completed: SE202

,event_id,time_ut,datetime_ut,x_km_sun,y_km_sun,z_km_sun,vx_kms_sun,vy_kms_sun,vz_kms_sun,x_km_moon,y_km_moon,z_km_moon,vx_kms_moon,vy_kms_moon,vz_kms_moon
0,SE1991Jan15A,22:00,1991-01-15 22:00:00,6.308259e+07,-1.219827e+08,-5.288898e+07,27.408563,11.822674,5.127322,166748.057695,-334228.862012,-147465.422969,0.880393,0.358851,0.254469
1,SE1991Jan15A,22:01,1991-01-15 22:01:00,6.308423e+07,-1.219820e+08,-5.288868e+07,27.408405,11.822981,5.127455,166800.879446,-334207.327256,-147450.153209,0.880332,0.358974,0.254523
2,SE1991Jan15A,22:02,1991-01-15 22:02:00,6.308588e+07,-1.219813e+08,-5.288837e+07,27.408247,11.823287,5.127588,166853.697507,-334185.785103,-147434.880184,0.880270,0.359098,0.254578
3,SE1991Jan15A,22:03,1991-01-15 22:03:00,6.308752e+07,-1.219806e+08,-5.288806e+07,27.408088,11.823593,5.127721,166906.511879,-334164.235553,-147419.603895,0.880209,0.359221,0.254632
4,SE1991Jan15A,22:04,1991-01-15 22:04:00,6.308916e+07,-1.219799e+08,-5.288775e+07,27.407930,11.823900,5.127853,166959.322559,-334142.678608,-147404.324342,0.880147,0.359344,0.254686


# Data Sources, Copyright, and Usage Rights

## Dataset A — NASA Five Millennium Catalog of Solar Eclipses

**Source:** NASA Goddard Space Flight Center (GSFC).

**Copyright:** Eclipse calculations by Fred Espenak and Jean Meeus.

**Usage Rights:** NASA explicitly permits reproduction of the eclipse data with appropriate attribution. The dataset can be used for academic research, machine learning, data analysis, and scientific visualization.

**Required Attribution:** "Eclipse Predictions by Fred Espenak and Jean Meeus (NASA's GSFC)."

**Official Source:** https://eclipse.gsfc.nasa.gov/SEpubs/5MKSE.html

---

## Dataset B — NASA Besselian Elements

**Source:** NASA Goddard Space Flight Center (GSFC).

**Copyright:** Eclipse calculations by Fred Espenak, with Besselian elements provided by Jean Meeus.

**Usage Rights:** Reproduction of the eclipse data is permitted with appropriate acknowledgment. The coefficients may be used for mathematical modeling, machine learning, and scientific research.

**Required Attribution:** "Eclipse Predictions by Fred Espenak and Jean Meeus (NASA's GSFC)."

**Official Source:** https://eclipse.gsfc.nasa.gov/SEpubs/5MKSE.html

---

## Dataset C — NASA Central Solar Eclipse Paths

**Source:** NASA Goddard Space Flight Center (GSFC).

**Copyright:** Eclipse calculations by Fred Espenak.

**Usage Rights:** NASA explicitly permits reproduction of the eclipse trajectory data with appropriate acknowledgment. The coordinates may be used for geographical visualization, model validation, and comparisons between predicted and reference eclipse paths.

**Required Attribution:** "Eclipse Predictions by Fred Espenak, NASA/GSFC."

**Official Source:** https://eclipse.gsfc.nasa.gov/SEpath/SEpath.html

---

## Dataset D — JPL Horizons API

**Source:** NASA Jet Propulsion Laboratory (JPL), California Institute of Technology (Caltech).

**Copyright:** NASA/JPL-Caltech. No specific Creative Commons license has been identified for the Horizons API output.

**Usage Rights:** Ephemeris data are publicly accessible for scientific applications. Automated access must comply with the JPL Fair Use Policy.

The policy requires reasonable request frequency, sequential API requests, appropriate identification through a User-Agent header, and caching of previously retrieved data whenever practical. Direct embedding of the API in a website is not permitted under the published NASA CORS policy.

**Recommended Attribution:** NASA/JPL Horizons On-Line Ephemeris System, Jet Propulsion Laboratory, California Institute of Technology.

**Official Source:** https://ssd.jpl.nasa.gov/horizons/

**API Documentation and Fair Use Policy:** https://ssd-api.jpl.nasa.gov/doc/

---

## General Usage Statement

The datasets are used for an independent academic machine learning project focused on predicting and visualizing solar eclipse shadow trajectories.

All original sources and contributors are acknowledged. The resulting models, predictions, and visualizations are independently developed and do not represent official NASA or JPL products.

The project does not claim endorsement, verification, or affiliation with NASA, JPL, or Caltech.
